In [12]:
from pathlib import Path
import pandas as pd
import plotly.express as px


In [13]:
DATA_PROCESSED = Path("../../data/processed")
PERSONAL_DIR = DATA_PROCESSED / "personal_atlas"

overlay = pd.read_parquet(PERSONAL_DIR / "personal_atlas_overlay.parquet")
song_league = pd.read_parquet(PERSONAL_DIR / "bts_song_league.parquet")
album_league = pd.read_parquet(PERSONAL_DIR / "bts_album_league.parquet")

overlay.shape, song_league.shape, album_league.shape

((381, 400), (344, 8), (86, 6))

In [14]:
top_songs = (
    overlay[overlay["personal_hours"] > 0]
    .sort_values("personal_hours", ascending=False)
    [["track_name", "album_name", "personal_hours", "personal_plays", "mastery_level"]]
    .head(20)
)

top_songs

,track_name,album_name,personal_hours,personal_plays,mastery_level
300,Tomorrow,Skool Luv Affair,3.837981,102,4
291,Tomorrow,Skool Luv Affair (Special Addition),3.837981,102,4
144,MIC Drop,Love Yourself 結 'Answer',2.375706,63,4
147,MIC Drop (Steve Aoki Remix) (Full Length Edition),Love Yourself 結 'Answer',2.375706,63,4
363,MIC Drop (feat. Desiigner) [Steve Aoki Remix],MIC Drop (feat. Desiigner) [Steve Aoki Remix],2.375706,63,4
176,MIC Drop,Love Yourself 承 'Her',2.375706,63,4
276,Let Me Know,Dark & Wild,2.312484,50,4
175,Pied Piper,Love Yourself 承 'Her',1.935010,55,3
239,House of Cards (Full Length Edition),The Most Beautiful Moment in Life: Young Forever,1.917402,39,3
150,FAKE LOVE,Love Yourself 轉 'Tear',1.790389,108,3


In [15]:
mastery_distribution = (
    overlay
    .groupby("mastery_level")
    .agg(
        songs=("track_name", "count"),
        total_hours=("personal_hours", "sum"),
    )
    .reset_index()
)

mastery_distribution

,mastery_level,songs,total_hours
0,0,68,0.000000
1,1,237,25.915316
2,2,33,22.570391
3,3,36,52.230623
4,4,7,19.491270


In [16]:
fig = px.bar(
    mastery_distribution,
    x="mastery_level",
    y="songs",
    title="Personal mastery distribution",
    labels={
        "mastery_level": "Mastery level",
        "songs": "Number of songs",
    },
)

fig.show()

In [17]:
top_albums = album_league.sort_values("hours_played", ascending=False).head(20)
top_albums

,master_metadata_album_artist_name,master_metadata_album_album_name,plays,hours_played,first_play,last_play
0,BTS,ARIRANG,272,7.338642,2026-03-21 15:27:06+00:00,2026-07-08 21:46:04+00:00
1,BTS,MAP OF THE SOUL : 7,219,6.942204,2024-08-23 14:34:25+00:00,2026-07-08 19:23:30+00:00
2,BTS,Love Yourself 轉 'Tear',226,5.121804,2024-08-29 18:39:12+00:00,2026-07-08 09:42:23+00:00
3,BTS,Love Yourself 承 'Her',156,4.840806,2024-07-24 18:45:38+00:00,2026-07-08 15:34:46+00:00
4,BTS,Skool Luv Affair,130,4.596744,2024-11-05 10:08:55+00:00,2026-07-08 20:53:29+00:00
5,BTS,Love Yourself 結 'Answer',182,4.549797,2024-09-04 19:24:41+00:00,2026-07-08 19:21:48+00:00
6,BTS,Wings,211,4.130166,2024-09-05 04:25:05+00:00,2026-07-08 19:23:32+00:00
7,BTS,Dark & Wild,111,3.737432,2025-04-23 07:50:23+00:00,2026-07-08 19:21:08+00:00
8,BTS,The Most Beautiful Moment in Life: Young Forever,133,3.581411,2024-11-02 12:28:14+00:00,2026-07-05 17:15:19+00:00
9,BTS,The Most Beautiful Moment in Life Pt.2,98,2.719907,2024-08-23 14:40:49+00:00,2026-07-08 09:24:58+00:00


In [18]:
fig = px.bar(
    top_albums.sort_values("hours_played"),
    x="hours_played",
    y="master_metadata_album_album_name",
    color="master_metadata_album_artist_name",
    orientation="h",
    title="Top BTS universe albums by listening hours",
    labels={
        "hours_played": "Hours played",
        "master_metadata_album_album_name": "Album",
        "master_metadata_album_artist_name": "Artist",
    },
)

fig.show()

In [19]:
[col for col in overlay.columns if col.lower() in ["x", "y", "umap_x", "umap_y"] or "x" == col.lower() or "y" == col.lower()]

[]

## Merge atlas coordinates

The Streamlit atlas loads `song_atlas.csv`, so this production export provides the existing two-dimensional semantic layout.

In [20]:
ATLAS_PATH = DATA_PROCESSED / "song_atlas.csv"
atlas = pd.read_csv(ATLAS_PATH)

coordinate_columns = [
    col
    for col in ["track_id", "x", "y", "cluster", "canonical_title", "is_primary_version"]
    if col in atlas.columns
]

print("Shape:", atlas.shape)
print("Relevant columns:", coordinate_columns)
atlas[coordinate_columns].head()

Shape: (381, 15)
Relevant columns: ['track_id', 'x', 'y', 'cluster', 'canonical_title', 'is_primary_version']


,track_id,x,y,cluster,canonical_title,is_primary_version
0,6nt3AoYjkaqXMZhypTBky1,-5.485134,4.084106,0,swim,False
1,7EytKcb3klVPpN5IW1sj1Y,-5.446604,4.568232,0,swim with rm,True
2,5dZLsPskKzph16LWo31uxL,-4.828399,4.061772,0,swim with jin,True
3,5AL5OrvyIMPqKjl9iw3xO5,-5.098581,4.094941,0,swim with suga,True
4,3MCJY7lXCHa0UNIjsAucaJ,-4.906363,4.174532,0,swim with jhope,True


In [21]:
atlas_coordinate_check = pd.Series(
    {
        "rows": len(atlas),
        "unique_track_ids": atlas["track_id"].nunique(),
        "duplicate_track_ids": atlas["track_id"].duplicated().sum(),
        "missing_x": atlas["x"].isna().sum(),
        "missing_y": atlas["y"].isna().sum(),
    }
)

atlas_coordinate_check

rows                   381
unique_track_ids       381
duplicate_track_ids      0
missing_x                0
missing_y                0
dtype: int64

In [22]:
atlas_columns = [
    col
    for col in ["track_id", "x", "y", "cluster", "canonical_title", "is_primary_version"]
    if col in atlas.columns and (col == "track_id" or col not in overlay.columns)
]
atlas_coordinates = atlas[atlas_columns].copy()

merge_validation = (
    "one_to_one"
    if overlay["track_id"].is_unique and atlas_coordinates["track_id"].is_unique
    else "many_to_one"
)

personal_map = overlay.merge(
    atlas_coordinates,
    on="track_id",
    how="left",
    validate=merge_validation,
    indicator=True,
)

merge_validation, personal_map.shape

('one_to_one', (381, 406))

In [23]:
valid_coordinates = personal_map[["x", "y"]].notna().all(axis=1)
unmatched = personal_map[personal_map["_merge"] == "left_only"]

merge_check = pd.Series(
    {
        "rows_before_merge": len(overlay),
        "rows_after_merge": len(personal_map),
        "coordinate_coverage_pct": valid_coordinates.mean() * 100,
        "unmatched_track_ids": unmatched["track_id"].nunique(),
        "duplicate_track_ids": personal_map["track_id"].duplicated().sum(),
        "rows_with_listening_history": personal_map["has_listening_history"].sum(),
        "rows_without_listening_history": (~personal_map["has_listening_history"]).sum(),
    }
)

merge_check

rows_before_merge                 381.0
rows_after_merge                  381.0
coordinate_coverage_pct           100.0
unmatched_track_ids                 0.0
duplicate_track_ids                 0.0
rows_with_listening_history       313.0
rows_without_listening_history     68.0
dtype: float64

In [24]:
unmatched[["track_id", "track_name", "album_name"]].head(10)

,track_id,track_name,album_name


In [25]:
personal_map = personal_map.drop(columns="_merge")

identity_columns = [
    "track_id",
    "track_name",
    "album_name",
    "release_year",
    "canonical_title",
    "is_primary_version",
    "x",
    "y",
    "cluster",
]
personal_columns = [
    "normalized_atlas_title",
    "personal_plays",
    "personal_hours",
    "personal_total_ms",
    "personal_first_play",
    "personal_last_play",
    "source_history_titles",
    "match_types",
    "min_fuzzy_score",
    "personal_rank",
    "mastery_level",
    "has_listening_history",
]
map_columns = [col for col in identity_columns + personal_columns if col in personal_map.columns]
personal_map = personal_map[map_columns].copy()

OUTPUT_PATH = PERSONAL_DIR / "personal_atlas_map.parquet"
personal_map.to_parquet(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)
print("Shape:", personal_map.shape)
print(f"Coordinate coverage: {valid_coordinates.mean():.1%}")

Saved: ../../data/processed/personal_atlas/personal_atlas_map.parquet
Shape: (381, 21)
Coordinate coverage: 100.0%


## Personal Atlas preview

In [26]:
preview_map = personal_map.assign(
    preview_size=personal_map["personal_plays"].pow(0.5).add(4)
)

fig = px.scatter(
    preview_map,
    x="x",
    y="y",
    color="personal_hours",
    size="preview_size",
    size_max=18,
    hover_name="track_name",
    hover_data={
        "x": False,
        "y": False,
        "preview_size": False,
        "album_name": True,
        "personal_hours": ":.2f",
        "personal_plays": True,
        "mastery_level": True,
    },
    title="Personal Atlas: listening intensity across the semantic map",
    labels={
        "album_name": "Album",
        "personal_hours": "Hours",
        "personal_plays": "Plays",
        "mastery_level": "Mastery level",
    },
    template="plotly_dark",
)

fig.update_traces(marker={"opacity": 0.8, "line": {"width": 0}})
fig.update_layout(
    height=700,
    margin={"l": 0, "r": 0, "t": 60, "b": 0},
    plot_bgcolor="#060817",
    paper_bgcolor="#060817",
    coloraxis_colorbar={"title": "Listening hours"},
)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False, scaleanchor="x", scaleratio=1)

fig.show()

## Conclusion

The production atlas coordinates are now joined to the personal listening overlay with complete merge coverage, and `personal_atlas_map.parquet` provides a visualization-ready dataset. The Personal Atlas can now support a heatmap or listening-intensity mode while preserving the semantic layout. Future reruns after the solo-discography expansion should improve personal-history coverage.